# 02 · Resultados para el artículo

Agrega las corridas de la carpeta compartida y genera tablas (CSV + LaTeX booktabs) y figuras (PDF + PNG a 300 dpi, ancho de columna IEEE).

- **Colab:** monta Drive y lee `IA_ProyectoFinal_RoCoLe/results`.
- **Local:** define `ROCOLE_RESULTS_ROOT` (por ejemplo, la carpeta sincronizada con Google Drive para escritorio) y `ROCOLE_DATA_ROOT` (para la figura de mapas).

Salidas en el repositorio: `results/aggregate/*.csv`, `results/tables/*.{csv,tex}`, `figures/*.{pdf,png}` y copia de los JSON en `results/runs/` (sin pesos ni mapas).

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Rosa-SDev/roya-cafe-cbam-vs-deit.git"
SHARED_DIR = Path("/content/drive/MyDrive/IA_ProyectoFinal_RoCoLe")  # carpeta compartida (acceso directo)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = Path("/content/codigo")
    if not REPO.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Repositorio:", REPO, "| commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
      capture_output=True, text=True).stdout.strip())

In [ ]:
import shutil
import pandas as pd
from src import report
from src.aggregate import aggregate, load_runs
from src.runs import all_runs, result_path

RESULTS_ROOT = SHARED_DIR / "results" if IN_COLAB else Path(os.environ.get("ROCOLE_RESULTS_ROOT", REPO / "results"))
DATA_ROOT = Path("/content/data") if IN_COLAB else Path(os.environ.get("ROCOLE_DATA_ROOT", REPO / "data" / "cache"))
FIG_DIR, TAB_DIR = REPO / "figures", REPO / "results" / "tables"

expected = all_runs()
missing = [r["name"] + " (" + r["experiment"] + ")" for r in expected if not result_path(RESULTS_ROOT, r).exists()]
print(f"Corridas encontradas: {len(expected) - len(missing)} de {len(expected)}")
if missing:
    print("Faltan:", *missing[:20], "..." if len(missing) > 20 else "", sep="\n  ")

In [ ]:
# Copia los JSON al repositorio (versionables; los .npz y .pt no se suben)
if RESULTS_ROOT.resolve() != (REPO / "results").resolve():
    for path in (RESULTS_ROOT / "runs").glob("E*/*.json"):
        dest = REPO / "results" / "runs" / path.parent.name / path.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, dest)

tables = aggregate(RESULTS_ROOT, REPO / "results" / "aggregate", reps=2000)
summary, boot, comp = tables["summary"], tables["bootstrap"], tables["comparisons"]
runs = load_runs(RESULTS_ROOT)
display(summary[["experiment", "config", "fraction", "n_seeds", "macro_f1_mean", "macro_f1_std", "accuracy_mean"]])

## Hipótesis: diferencias pareadas (bootstrap por planta, IC95)

`point` es la diferencia de medias entre semillas (A − B); `p_le_0` es la fracción de réplicas bootstrap con diferencia ≤ 0 (cercana a 0 respalda A > B).

In [ ]:
pd.set_option("display.max_colwidth", 60)
display(comp[comp.metric.isin(["macro_f1"]) | comp.hypothesis.eq("H4")].round(4))

## Tablas

In [ ]:
report.setup_style()
t1 = report.table_e1(summary, boot)
report.save_table(t1, TAB_DIR, "tabla_e1", "Resultados en prueba (E1): media $\\pm$ desviación entre 5 semillas e IC95 por bootstrap por planta.", "tab:e1")
t2 = report.table_e2(summary, boot)
report.save_table(t2, TAB_DIR, "tabla_e2_ablacion", "Ablación de CBAM en la CNN entrenada desde cero (E2).", "tab:e2")
t3 = report.table_params_times(summary)
report.save_table(t3, TAB_DIR, "tabla_parametros_tiempos", "Parámetros y tiempos (GPU T4; inferencia con lote 1).", "tab:params")
comp.round(4).to_csv(TAB_DIR / "comparaciones_hipotesis.csv", index=False)
display(t1); display(t2); display(t3)

## Figuras

In [ ]:
report.fig_f1_bars(summary, FIG_DIR)
report.fig_data_curves(summary, FIG_DIR)
report.fig_confusions(runs, summary, FIG_DIR)
report.fig_recall_per_class(summary, FIG_DIR)
report.fig_in_leaf(summary, FIG_DIR)
if (DATA_ROOT / "rocole_256.npz").exists():
    from src.data import load_cache
    cache = load_cache(DATA_ROOT)
    report.fig_attention_examples(RESULTS_ROOT, runs, cache["images"], cache["names"], FIG_DIR)
else:
    print("Sin caché de imágenes: se omite la figura de mapas de atención")
print(sorted(p.name for p in FIG_DIR.glob("*.pdf")))

In [ ]:
from IPython.display import Image, display as show
for name in ["f1_macro_por_configuracion", "curvas_datos_f1", "matrices_confusion_mejor_por_familia",
             "sensibilidad_por_clase", "fraccion_atencion_hoja", "ejemplos_mapas_atencion"]:
    if (FIG_DIR / f"{name}.png").exists():
        show(Image(filename=str(FIG_DIR / f"{name}.png"), width=700))